# Regressions

This notebook demonstrates various regressions and is adopted from Michael Ekstrand.

## Setup

We're going to import our modules.  Of particular note is `statsmodels.formula.api`, which gives us the regression APIs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import scipy.stats as sps

%pip install statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf

We're going to use the Penguins data:

In [ ]:
penguins = pd.read_csv('penguins.csv')

In [ ]:
penguins.head()

I'm also going to create a utility function for plotting lines:

In [ ]:
def plot_line(intercept, slope, xmin, xmax, **kwargs):
    xs = np.linspace(xmin, xmax, 100)
    ys = intercept + slope * xs
    plt.plot(xs, ys, **kwargs)

## Explore Correlations

The Seaborn [`pairplot`](https://seaborn.pydata.org/archive/0.10/generated/seaborn.pairplot.html#seaborn.pairplot) function shows a *scatterplot matrix*, allowing us to view possible correlations:

In [ ]:
sns.pairplot(penguins[['species', 'sex', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']], hue='species')

Let's also look at the distribution of sex and species, to see if we have sex imbalances between species:

In [ ]:
sns.countplot(x='species', hue='sex', data=penguins)

Nope!

Now a correlation matrix between our various numeric variables:

In [ ]:
penguins.corr()

The `.corr` method on a data frame will compute correlations between all pairs of numeric variables, and show them in a table.  The diagonal is 1, because a variable is always correlated with itself.

### Dummy Variable Demo

Here's the code that demonstrates dummy-coding from the Categorical Predictors lecture:

In [ ]:
pd.get_dummies(penguins['species'])

And dropping the first level:

In [ ]:
pd.get_dummies(penguins['species'], drop_first=True)

## Flipper Regression

From the pair plot, the flipper length and body mass look like good candidates for regression.  Let's show them with a `regplot`:

In [ ]:
sns.regplot('flipper_length_mm', 'body_mass_g', data=penguins, line_kws={'color': 'orange'})

Now we're going to fit a logistic regression.

The first step is to create the OLS model.  This has two inputs:

1. The formula specifying the regression model (in this case, predicting body mass with flipper length)
2. The data to train the model on

Let's do it:

In [ ]:
bm_mod = smf.ols('body_mass_g ~ flipper_length_mm', data=penguins)

Once we have *created* the model, we need to *fit* it.  The [`fit`](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.fit.html#statsmodels.regression.linear_model.OLS.fit) method will return an object containing the results of fitting the model, such as its paramters; it can be summarized with `summary`:

In [ ]:
bmf = bm_mod.fit()
bmf.summary()

Now we need to check our residual assumptions.  There are two plots to create:

* Residuals vs. Fitted 
* Residual Q-Q

I'm going to create a function that plots them both:

In [ ]:
def plot_lm_diag(fit):
    "Plot linear fit diagnostics"
    sns.regplot(x=fit.fittedvalues, y=fit.resid)
    plt.xlabel('Fitted')
    plt.ylabel('Residuals')
    plt.title('Residuals vs. Fitted')
    plt.show()

    sm.qqplot(fit.resid, fit=True, line='45')
    plt.title('Residuals')
    plt.show()

And now use it:

In [ ]:
plot_lm_diag(bmf)

### Standardizing Variables

We can also *standardize* our variables before fitting a model.  This converts variable values into *z*-scores, such that

$$z_i = \frac{x_i - \bar{x}}{s}$$

Z-standardized (or z-normalized) variables have a mean of 0 and a standard deviation of 1 (and since $1^2 = 1$, the variance is also 1).  The result is that model coefficients are in units of standard deviations.

Let's write a function to convert a series to its *z*-scores:

In [ ]:
def zscore(xs):
    xbar = xs.mean()
    s = xs.std()
    return (xs - xbar) / s

And create standardized versions of our variables:

In [ ]:
penguin_std = pd.DataFrame({
    'species': penguins['species'],
    'sex': penguins['sex'],
    'mass': zscore(penguins['body_mass_g']),
    'flipper': zscore(penguins['flipper_length_mm']),
    'bill_len': zscore(penguins['bill_length_mm']),
    'bill_depth': zscore(penguins['bill_depth_mm'])
})
penguin_std

And now we'll fit and describe a model using these:

In [ ]:
bm_mod = smf.ols('mass ~ flipper', data=penguin_std)
bmf = bm_mod.fit()
bmf.summary()

The residual plots will be identical to the unstandardized model, except for the scales of the axes, because all we did was rescale and recenter the variables:

In [ ]:
plot_lm_diag(bmf)

### Mass and Bill Length

Let's do a regression we don't expect to work well: mass vs. bill length.

First, we'll fit the regression:

In [ ]:
bm_mod = smf.ols('mass ~ bill_len', data=penguin_std)
bmf = bm_mod.fit()
bmf.summary()

$R^2$ is much lower than the previous models, indicating this model explains much less of the variance.

Let's look at our diagnostic plots:

In [ ]:
plot_lm_diag(bmf)

There's some noticable non-normality at the tails of the residuals, but it isn't bad. The residuals, however, exhibit notable heteroskedasticity,.

## Multivariate Regression


### Per-Species Intercepts

Let's add the species as a predictor, so we have per-species intercepts for the variable:

In [ ]:
bm_mod = smf.ols('mass ~ flipper + species', data=penguin_std)
bmf = bm_mod.fit()
bmf.summary()

Species is automatically dummy-coded with the first level dropped.  We therefore get the following intercepts:

- Adelie: `Intercept`
- Chinstrap: `Intercept + species[T.Chinstrap]`
- Gentoo: `Intercept + species[T.Gentoo]`

Let's check fit:

In [ ]:
plot_lm_diag(bmf)

And we'll plot the different lines on top of a scatterplot:

In [ ]:
sns.scatterplot('flipper', 'mass', hue='species', data=penguin_std)
plot_line(bmf.params['Intercept'], bmf.params['flipper'], -2, 2, label='Adelie')
plot_line(bmf.params['Intercept'] + bmf.params['species[T.Chinstrap]'],
          bmf.params['flipper'],
          -2, 2, label='Chinstrap')
plot_line(bmf.params['Intercept'] + bmf.params['species[T.Gentoo]'],
          bmf.params['flipper'],
          -2, 2, label='Gentoo')


### Per-Species Slopes and Intercepts

Now let's add an interaction term between flipper and species.

We do this with `flipper * species`, which expands to `flipper + species + flipper:species`:

In [ ]:
bm_mod = smf.ols('mass ~ flipper * species', data=penguin_std)
bmf = bm_mod.fit()
bmf.summary()

AIC is improving slightly, not a lot.

Look at assumption checks:

In [ ]:
plot_lm_diag(bmf)

Getting better, but still some curve on the Q-Q and doesn't quite look homoskedastic.  (Watch the video for more explanation.)

Let's plot these different lines:

In [ ]:
sns.scatterplot('flipper', 'mass', hue='species', data=penguin_std)
plot_line(bmf.params['Intercept'], bmf.params['flipper'], -2, 2, label='Adelie')
plot_line(bmf.params['Intercept'] + bmf.params['species[T.Chinstrap]'],
          bmf.params['flipper'] + bmf.params['flipper:species[T.Chinstrap]'],
          -2, 2, label='Chinstrap')
plot_line(bmf.params['Intercept'] + bmf.params['species[T.Gentoo]'],
          bmf.params['flipper'] + bmf.params['flipper:species[T.Gentoo]'],
          -2, 2, label='Gentoo')


### Sexual Dimorphism

Now let's look at penguin sex as a predictor, and we'll keep the flipper:species interaction:

In [ ]:
bm_mod = smf.ols('mass ~ flipper + species * sex + flipper:species', data=penguin_std)
bmf = bm_mod.fit()
bmf.summary()

This is good, but some non-significant relationships. In particular, the flipper/species interaction disappeared.

Assumption checks:

In [ ]:
plot_lm_diag(bmf)

That is looking pretty good.

Drop interaction, due to non-significant coefficients:

In [ ]:
bm_mod = smf.ols('mass ~ flipper + species * sex', data=penguin_std)
bmf = bm_mod.fit()
bmf.summary()

This is definitely the best AIC we've seen. Let's look at those assumption check plots:

In [ ]:
plot_lm_diag(bmf)

That looks quite good.

## Bootstrap

Now let's bootstrap some of these model parameters.  We'll start with a function that will compute bootstrap samples of the *rows* of a data frame, and pass them to a function to get a statistic:

In [ ]:
def boot_rows(df, stat, nboot=10000):
    n = len(df)
    obs = stat(df)
    boots = [stat(df.sample(n=n, replace=True)) for i in range(nboot)]
    return obs, np.quantile(boots, [0.025, 0.975])

Now we'll write a function to train our full OLS model, and return the flipper length coefficient:

In [ ]:
def flipper_coefficient(df):
    bm_mod = smf.ols('mass ~ flipper + species * sex', data=df)
    bmf = bm_mod.fit()
    return bmf.params['flipper']

And we'll bootstrap the flipper length coefficient:

In [ ]:
boot_rows(penguin_std, flipper_coefficient, nboot=1000)

That matches the CIs from the model itself pretty closely!

### Simultaneous Bootstrap of Multiple Parameters

In the video, I said you could do this. Here's the code.

First, we need a `boot_rows` function that can deal with the `stat` function returning a series of multiple statistics:

In [ ]:
def boot_rows(df, stat, nboot=10000):
    n = len(df)
    obs = stat(df)
    boots = [stat(df.sample(n=n, replace=True)) for i in range(nboot)]
    if isinstance(obs, pd.Series):
        # we have multiple statistics - stack them into a frame
        boots = pd.DataFrame.from_records(boots)
        # compute confidence intervals - columns will be params, and rows CI bounds
        conf = boots.quantile([0.025, 0.975])
        # transpose will put the params on rows
        # join with frame-ified version of the original observations
        return obs.to_frame('value').join(conf.transpose())
    else:
        return obs, np.quantile(boots, [0.025, 0.975])

And a function to return the parameters from a trained model:

In [ ]:
def flipper_params(df):
    bm_mod = smf.ols('mass ~ flipper + species * sex', data=df)
    bmf = bm_mod.fit()
    return bmf.params

And let's bootstrap all of this:

In [ ]:
boot_rows(penguin_std, flipper_params, nboot=1000)